In [1]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("team-ai/spam-text-message-classification")

print("Path to dataset files:", path)

100%|██████████| 208k/208k [00:00<00:00, 745kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/team-ai/spam-text-message-classification/versions/1


In [2]:

import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [8]:
import os

for root, dirs, files in os.walk(path):
    for name in files:
        print(os.path.join(root, name))

/root/.cache/kagglehub/datasets/team-ai/spam-text-message-classification/versions/1/SPAM text message 20170820 - Data.csv


In [10]:
import os
for dirname, _, filenames in os.walk('/kaggle/input/ag-news-classification-dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv(f"{path}/SPAM text message 20170820 - Data.csv")

# Split the data into training and testing sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Show first 5 rows of train set
print("Train set preview:")
display(train_df.head())

# Show first 5 rows of test set
print("Test set preview:")
display(test_df.head())

# Check the shape of the datasets
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# See column names
print("Columns:", train_df.columns.tolist())

Train set preview:


,Category,Message
1978,spam,Reply to win £100 weekly! Where will the 2006 ...
3989,ham,Hello. Sort of out in town already. That . So ...
3935,ham,How come guoyang go n tell her? Then u told her?
4078,ham,Hey sathya till now we dint meet not even a si...
4086,spam,Orange brings you ringtones from all time Char...


Test set preview:


,Category,Message
3245,ham,Squeeeeeze!! This is christmas hug.. If u lik ...
944,ham,And also I've sorta blown him off a couple tim...
1044,ham,Mmm thats better now i got a roast down me! i...
2484,ham,Mm have some kanji dont eat anything heavy ok
812,ham,So there's a ring that comes with the guys cos...


Train shape: (4457, 2)
Test shape: (1115, 2)
Columns: ['Category', 'Message']


In [14]:
import os, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

In [15]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 32
NUM_EPOCHS = 3
LR = 1e-3
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

Device: cuda


In [17]:
train_df, test_df

(     Category                                            Message
 1978     spam  Reply to win £100 weekly! Where will the 2006 ...
 3989      ham  Hello. Sort of out in town already. That . So ...
 3935      ham   How come guoyang go n tell her? Then u told her?
 4078      ham  Hey sathya till now we dint meet not even a si...
 4086     spam  Orange brings you ringtones from all time Char...
 ...       ...                                                ...
 3772      ham  Hi, wlcome back, did wonder if you got eaten b...
 5191      ham                             Sorry, I'll call later
 5226      ham      Prabha..i'm soryda..realy..frm heart i'm sory
 5390      ham                         Nt joking seriously i told
 860       ham            Did he just say somebody is named tampa
 
 [4457 rows x 2 columns],
      Category                                            Message
 3245      ham  Squeeeeeze!! This is christmas hug.. If u lik ...
 944       ham  And also I've sorta blown him of

In [20]:
train_texts = (train_df["Message"].fillna("") + ". " + train_df["Category"].fillna("")).tolist()
test_texts  = (test_df["Message"].fillna("") + ". " + test_df["Category"].fillna("")).tolist()

In [21]:
train_labels = (train_df["Category"].apply(lambda x: 1 if x == "spam" else 0)).tolist()
test_labels  = (test_df["Category"].apply(lambda x: 1 if x == "spam" else 0)).tolist()

label_names = ["ham", "spam"]
num_labels = len(label_names)

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        t = self.texts[idx]
        enc = self.tokenizer(
            t,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(train_texts, train_labels, tokenizer, MAX_LEN)
test_ds  = TextDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [23]:
bert = AutoModel.from_pretrained(MODEL_NAME)
for p in bert.parameters():
    p.requires_grad = False
bert.to(DEVICE)
bert.eval()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [24]:
def compute_embeddings(dataloader, model, device):
    all_embs, all_labels = [], []
    for batch in tqdm(dataloader, desc="Embedding"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        cls_emb = outputs.last_hidden_state[:,0,:]  # CLS token
        all_embs.append(cls_emb.cpu().numpy())
        all_labels.append(batch["labels"].cpu().numpy())
    return np.concatenate(all_embs), np.concatenate(all_labels)

**Approach 1: Frozen BERT + Trainable Head**

In [25]:
class FrozenBertHead(nn.Module):
    def __init__(self, bert_model, hidden_size, num_labels):
        super().__init__()
        self.bert = bert_model
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size//2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size//2, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            cls = outputs.last_hidden_state[:,0,:]
        return self.classifier(cls)

In [ ]:
hidden_size = bert.config.hidden_size
model_head = FrozenBertHead(bert, hidden_size, num_labels).to(DEVICE)

optimizer = torch.optim.AdamW(model_head.classifier.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

In [ ]:
print("\n=== Training Approach 1 (Frozen BERT + Head) ===")
for epoch in range(NUM_EPOCHS):
    model_head.train()
    running_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = model_head(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1} loss: {running_loss/len(train_loader):.4f}")


=== Training Approach 1 (Frozen BERT + Head) ===


Epoch 1/3:   0%|          | 0/3750 [00:00<?, ?it/s]

Epoch 1 loss: 0.3337


Epoch 2/3:   0%|          | 0/3750 [00:00<?, ?it/s]

Epoch 2 loss: 0.2927


Epoch 3/3:   0%|          | 0/3750 [00:00<?, ?it/s]

Epoch 3 loss: 0.2757


In [ ]:
model_head.eval()
preds, true = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Eval A1"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model_head(input_ids, attention_mask)
        preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
        true.extend(labels.cpu().tolist())
print("\n=== Report: Approach 1 ===")
print(classification_report(true, preds, target_names=label_names, digits=4))

Eval A1:   0%|          | 0/238 [00:00<?, ?it/s]


=== Report: Approach 1 ===
              precision    recall  f1-score   support

       World     0.9214    0.9253    0.9233      1900
      Sports     0.9726    0.9732    0.9729      1900
    Business     0.8680    0.8753    0.8716      1900
    Sci/Tech     0.8816    0.8700    0.8758      1900

    accuracy                         0.9109      7600
   macro avg     0.9109    0.9109    0.9109      7600
weighted avg     0.9109    0.9109    0.9109      7600



**BERT embeddings + Logistic Regression**

In [ ]:
print("\n=== Approach 2: Extract embeddings ===")
train_embs, train_lbls = compute_embeddings(train_loader, bert, DEVICE)
test_embs, test_lbls = compute_embeddings(test_loader, bert, DEVICE)

scaler = StandardScaler()
train_embs = scaler.fit_transform(train_embs)
test_embs  = scaler.transform(test_embs)

clf = LogisticRegression(max_iter=1000, multi_class="multinomial", solver="saga", n_jobs=-1)
clf.fit(train_embs, train_lbls)
preds2 = clf.predict(test_embs)

print("\n=== Report: Approach 2 ===")
print(classification_report(test_lbls, preds2, target_names=label_names, digits=4))



=== Approach 2: Extract embeddings ===


Embedding:   0%|          | 0/3750 [00:00<?, ?it/s]

Embedding:   0%|          | 0/238 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



=== Report: Approach 2 ===
              precision    recall  f1-score   support

       World     0.9130    0.9111    0.9120      1900
      Sports     0.9611    0.9742    0.9676      1900
    Business     0.8689    0.8616    0.8652      1900
    Sci/Tech     0.8765    0.8737    0.8751      1900

    accuracy                         0.9051      7600
   macro avg     0.9048    0.9051    0.9050      7600
weighted avg     0.9048    0.9051    0.9050      7600



**Approach 3: Cosine Similarity with label descriptions**

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

label_texts = [
    "world news and international events",
    "sports, games, competitions, and athletes",
    "business, finance, markets, economy",
    "science and technology, research and innovation"
]
label_embs = model.encode(label_texts, normalize_embeddings=True)
test_embs = model.encode(test_texts, normalize_embeddings=True)

sims = cosine_similarity(test_embs, label_embs)
preds3 = np.argmax(sims, axis=1)

print(classification_report(test_lbls, preds3, target_names=label_names, digits=4))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

              precision    recall  f1-score   support

       World     0.6061    0.7695    0.6781      1900
      Sports     0.8017    0.8258    0.8136      1900
    Business     0.6873    0.5553    0.6143      1900
    Sci/Tech     0.6285    0.5611    0.5929      1900

    accuracy                         0.6779      7600
   macro avg     0.6809    0.6779    0.6747      7600
weighted avg     0.6809    0.6779    0.6747      7600

